# C3: DSPy + GEPA — Tối ưu Prompt tự động cho GRACE Enhanced

**Pipeline tối ưu hoá:** Genetic-Pareto Prompt Optimization cho bài toán phát hiện lỗ hổng bảo mật.

## Tổng quan

Thay vì viết prompt thủ công (hand-crafted), ta dùng **DSPy + GEPA** để:
1. **DSPy**: Định nghĩa pipeline phát hiện lỗ hổng như một *chương trình có tham số*. Các tham số chính là nội dung của các thành phần prompt (Pb, Pi, Pd).
2. **GEPA** (Genetic-Pareto): Dùng LLM làm *giám khảo phản chiếu* để đọc trace lỗi, đề xuất cải tiến prompt, và chọn lọc theo Pareto frontier — *không cần gradient*.

## Luồng công việc (2 pha):

```
[Pha 1 — Offline Optimization]
  VAL SET (labeled) → DSPy Program → GEPA → Compiled Program (optimized_prompt.json)
  
[Pha 2 — Online Evaluation]
  TEST SET → load optimized_prompt.json → GRACE Enhanced + Optimized Prompt → Metrics (MCC, F1...)
```

## Cấu trúc notebook:
| Bước | Nội dung |
|------|---------|
| 1 | Cài đặt DSPy và cấu hình LM |
| 2 | Load dữ liệu (Val set + demonstrations từ RAG pipeline) |
| 3 | Định nghĩa DSPy Signature và Module |
| 4 | Xây dựng CWE-aware Metric |
| 5 | Chạy GEPA Optimization trên Val Set |
| 6 | Lưu Compiled Program |
| 7 | Evaluate trên Test Set với Optimized Prompt |
| 8 | So sánh kết quả (Baseline vs GRACE vs GRACE+GEPA) |

> **Lưu ý:** Notebook này phải chạy *sau* khi đã chạy xong `setup_and_run_wsl.ipynb`
> (đã có `devign_test_processed.json` và `bge_vectors_cache.npz`).


## Bước 1: Cài đặt DSPy và cấu hình LM

DSPy hỗ trợ nhiều LM backends. Ta sẽ cấu hình **Vertex AI (Qwen3-Coder)** làm `task_lm` (chạy task chính)
và dùng cùng model làm `reflection_lm` (GEPA dùng để phản chiếu và cải tiến prompt).


In [9]:
from tqdm import tqdm\n
%pip install dspy>=2.6 google-cloud-aiplatform scikit-learn pandas tqdm rank_bm25 FlagEmbedding qdrant-client tree-sitter tree-sitter-c


Note: you may need to restart the kernel to use updated packages.


## Bước 2: Cấu hình DSPy Language Model (Vertex AI)

GEPA cần 2 model roles:
- **`task_lm`**: Chạy task chính (phân loại vulnerability). Dùng Qwen3-Coder-480B.
- **`reflection_lm`**: LLM phản chiếu, đọc trace lỗi và đề xuất cải tiến prompt. 
  Dùng cùng Qwen3-Coder hoặc model mạnh hơn.

> Nếu bước này lỗi, chạy: `gcloud auth application-default login` trong WSL Terminal.


In [10]:
import random
import dspy
import vertexai
from vertexai.generative_models import GenerativeModel
import hashlib
import json
import os
import pickle

CACHE_FILE = "gepa_lm_cache.pkl"
CACHE_SAVE_INTERVAL = 10  # Lưu vào file sau mỗi 10 calls

class VertexGenerativeLM(dspy.LM):
    def __init__(self, model_name, project="grace-enhanced", location="global", **kwargs):
        super().__init__(model=model_name)
        vertexai.init(project=project, location=location)
        self.model = GenerativeModel(model_name)
        self.kwargs = {
            "temperature": kwargs.get("temperature", 0.0),
            "max_output_tokens": kwargs.get("max_tokens", 1000)
        }
        self.history = []
        
        # Load cache từ .pkl nếu có
        self.cache_dict = {}
        if os.path.exists(CACHE_FILE):
            try:
                with open(CACHE_FILE, "rb") as f:
                    self.cache_dict = pickle.load(f)
                print(f"[CACHE] Đã load {len(self.cache_dict)} entries từ {CACHE_FILE}")
            except Exception as e:
                print(f"[CACHE] Lỗi khi đọc {CACHE_FILE}: {e}")
                
        self.unsaved_calls = 0

    def __call__(self, prompt=None, messages=None, **kwargs):
        if messages:
            prompt = "\n".join([f"{m['role']}: {m['content']}" for m in messages])
            
        cache_key_str = json.dumps({"prompt": prompt, "kwargs": self.kwargs}, sort_keys=True)
        cache_key = hashlib.md5(cache_key_str.encode("utf-8")).hexdigest()
        
        if cache_key in self.cache_dict:
            response_text = self.cache_dict[cache_key]
            self.history.append({
                "prompt": prompt, "response": response_text, "kwargs": kwargs, "cached": True
            })
            return [response_text]
        
        # Nếu chưa có, gọi API
        response = self.model.generate_content(prompt, generation_config=self.kwargs)
        
        # Lưu vào dict bộ nhớ
        self.cache_dict[cache_key] = response.text
        self.unsaved_calls += 1
        
        # Lưu xuống .pkl sau mỗi CACHE_SAVE_INTERVAL calls
        if self.unsaved_calls >= CACHE_SAVE_INTERVAL:
            with open(CACHE_FILE, "wb") as f:
                pickle.dump(self.cache_dict, f)
            self.unsaved_calls = 0
        
        self.history.append({
            "prompt": prompt, "response": response.text, "kwargs": kwargs, "cached": False
        })
        return [response.text]

QWEN_MODEL = "publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas"
task_lm = VertexGenerativeLM(QWEN_MODEL)
reflection_lm = VertexGenerativeLM(QWEN_MODEL)

dspy.settings.configure(lm=task_lm)

print(f"[OK] DSPy configured with custom VertexGenerativeLM (Pickle Cache Enabled)")
print(f"     Task LM: {QWEN_MODEL}")


[OK] DSPy configured with custom VertexGenerativeLM (Pickle Cache Enabled)
     Task LM: publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas


## Bước 3: Load dữ liệu

Load **Val set** (dùng để optimize prompt) và **Test set** (dùng để evaluate sau khi optimize).

Dữ liệu lấy từ output của `setup_and_run_wsl.ipynb`:
- `devign_test_processed.json`: Test set đã có demonstrations từ Stratified Retrieval (C2)
- `function.json`: Toàn bộ dataset để chia lại val set

> **Quan trọng:** Val set KHÔNG được dùng để train/evaluate kết quả cuối cùng.
> Chỉ dùng để GEPA "học" cách cải tiến prompt.


In [11]:
# === Load toàn bộ dataset để lấy val set ===
if not os.path.exists("function.json"):
    if os.path.exists("../Grace-Code-Based/function.json"):
        import shutil
        shutil.copy("../Grace-Code-Based/function.json", "function.json")
        print("[OK] Copied function.json from Grace-Code-Based")
    else:
        raise FileNotFoundError("Cần chạy setup_and_run_wsl.ipynb trước!")

with open("function.json", "r") as f:
    all_data = json.load(f)

# Chia lại với cùng seed như notebook chính
random.seed(42)
indices = list(range(len(all_data)))
random.shuffle(indices)

n = len(all_data)
n_train = int(n * 0.8)
n_val = int(n * 0.1)

val_data = [all_data[i] for i in indices[n_train:n_train + n_val]]
print(f"Val set: {len(val_data)} examples")
print(f"Val vuln ratio: {sum(d['target'] for d in val_data)/len(val_data):.2%}")

# === Load test set (đã có demonstrations từ RAG pipeline) ===
if not os.path.exists("devign_test_processed.json"):
    raise FileNotFoundError("Cần chạy setup_and_run_wsl.ipynb trước để có devign_test_processed.json!")

with open("devign_test_processed.json", "r") as f:
    test_processed = json.load(f)
print(f"Test set: {len(test_processed)} examples (với stratified demonstrations)")


Val set: 2731 examples
Val vuln ratio: 46.36%
Test set: 2733 examples (với stratified demonstrations)


## Bước 4: Chuẩn bị Demonstrations cho Val Set

Val set cần demonstrations để GEPA có thể optimize prompts có chứa few-shot examples.

Ta sẽ dùng lại **Tri-signal RAG pipeline** (từ notebook chính) để lấy demonstrations cho val set,
hoặc dùng cache nếu đã có.

> Đây là một val set nhỏ hơn (dùng tối đa `MAX_VAL_SAMPLES` samples để tiết kiệm API calls).


In [12]:
from tqdm import tqdm\n
MAX_VAL_SAMPLES = 200   # Số val examples dùng để optimize (tiết kiệm API)
VAL_SEED = 42

random.seed(VAL_SEED)
val_subset = random.sample(val_data, min(MAX_VAL_SAMPLES, len(val_data)))
print(f"Val subset: {len(val_subset)} examples (từ {len(val_data)} tổng cộng)")
print(f"Val subset vuln ratio: {sum(d['target'] for d in val_subset)/len(val_subset):.2%}")

# === Load demonstrations cho val set (dùng simple retrieval từ train set) ===
val_demos_cache = "val_demos_cache.pkl"
if os.path.exists(val_demos_cache):
    print(f"Loading val demonstrations từ cache...")
    with open(val_demos_cache, "rb") as f:
        val_with_demos = pickle.load(f)
else:
    # Simple retrieval: dùng BM25 để tìm demonstrations từ train set
    # (Cần có rank_bm25 và train data đã xử lý)
    try:
        from rank_bm25 import BM25Okapi

        # Load train data
        train_data_raw = [all_data[i] for i in indices[:n_train]]
        train_codes = [d["func"] for d in train_data_raw]
        train_labels = [d["target"] for d in train_data_raw]

        # Chia pool theo nhãn
        vuln_pool = [(train_codes[i], "Vulnerable")
                     for i in range(len(train_codes)) if train_labels[i] == 1]
        nonvuln_pool = [(train_codes[i], "Non-vulnerable")
                        for i in range(len(train_codes)) if train_labels[i] == 0]

        # BM25 simple retrieval cho val
        bm25_all = BM25Okapi([c.split() for c, _ in vuln_pool + nonvuln_pool])
        all_pool = vuln_pool + nonvuln_pool

        val_with_demos = []
        for item in tqdm(val_subset, desc="Val retrieval"):
            tokens = item["func"].split()
            scores = bm25_all.get_scores(tokens)
            ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)

            # Lấy top-1 từ mỗi nhóm
            demos = []
            found_vuln, found_nonvuln = False, False
            for idx, score in ranked:
                code_str, label = all_pool[idx]
                if not found_vuln and label == "Vulnerable":
                    demos.append({"code": code_str, "label": label})
                    found_vuln = True
                elif not found_nonvuln and label == "Non-vulnerable":
                    demos.append({"code": code_str, "label": label})
                    found_nonvuln = True
                if found_vuln and found_nonvuln:
                    break

            val_with_demos.append({
                "func": item["func"],
                "target": item["target"],
                "demonstrations": demos,
                "node": "",  # Val set không cần CPG (Joern chậm)
                "edge": "",
            })

        with open(val_demos_cache, "wb") as f:
            pickle.dump(val_with_demos, f)
        print(f"[OK] Saved val demonstrations to {val_demos_cache}")

    except ImportError:
        # Fallback: dùng val set không có demonstrations
        print("[WARN] rank_bm25 chưa install, dùng val set không có demonstrations")
        val_with_demos = [
            {"func": d["func"], "target": d["target"],
             "demonstrations": [], "node": "", "edge": ""}
            for d in val_subset
        ]

print(f"[OK] Val set with demonstrations: {len(val_with_demos)} examples")


Val subset: 200 examples (từ 2731 tổng cộng)
Val subset vuln ratio: 52.00%
Loading val demonstrations từ cache...
[OK] Val set with demonstrations: 200 examples


## Bước 5: Định nghĩa DSPy Signature và Module

**Signature** mô tả I/O của task. Đây là nơi DSPy/GEPA sẽ tối ưu hóa **instructions** (prompt instructions).

**Module** là pipeline tính toán — ở đây ta dùng `dspy.ChainOfThought` để LLM có thể lý giải
trước khi đưa ra kết luận (Vulnerable/Non-vulnerable).

### 3 thành phần prompt được tối ưu (theo proposal):
- **Pb** (Base instruction): Nhiệm vụ tổng quát của LLM
- **Pi** (Code analysis instruction): Hướng dẫn phân tích code/graph  
- **Pd** (Demonstration instruction): Hướng dẫn học từ examples


In [13]:
class VulnDetectionSignature(dspy.Signature):
    """You are an expert security researcher and C/C++ programmer.
    Your task is to analyze a C/C++ function and determine if it contains security vulnerabilities.
    Study the provided code carefully, including its graph structure and similar examples.
    Reason step by step, then output your final verdict as exactly 'Vulnerable' or 'Non-vulnerable'."""

    # === Inputs ===
    code: str = dspy.InputField(
        desc="The C/C++ function source code to analyze for vulnerabilities."
    )
    graph_nodes: str = dspy.InputField(
        desc="Node information from the Code Property Graph (CPG) showing variable types, "
             "function calls, and program structure.",
        default=""
    )
    graph_edges: str = dspy.InputField(
        desc="Edge information from the CPG showing control flow, data flow, "
             "and call dependencies.",
        default=""
    )
    demonstrations: str = dspy.InputField(
        desc="Similar code examples with known vulnerability labels for reference. "
             "Use these examples to understand patterns.",
        default=""
    )

    # === Outputs ===
    reasoning: str = dspy.OutputField(
        desc="Step-by-step security analysis: identify suspicious patterns, "
             "data flows, and potential vulnerabilities. Reference CWE categories if applicable."
    )
    verdict: str = dspy.OutputField(
        desc="Final verdict: exactly 'Vulnerable' or 'Non-vulnerable'. No other text."
    )


class GRACEEnhancedDetector(dspy.Module):
    """GRACE Enhanced vulnerability detector với DSPy."""

    def __init__(self):
        super().__init__()
        self.detect = dspy.ChainOfThought(VulnDetectionSignature)

    def forward(self, code, graph_nodes="", graph_edges="", demonstrations=""):
        return self.detect(
            code=code[:4000],          # Truncate để fit context
            graph_nodes=graph_nodes[:1500],
            graph_edges=graph_edges[:1500],
            demonstrations=demonstrations[:3000],
        )


# Khởi tạo detector
detector = GRACEEnhancedDetector()
print("[OK] GRACEEnhancedDetector defined")
print(f"     Signature: {VulnDetectionSignature.__doc__.strip()[:100]}...")


[OK] GRACEEnhancedDetector defined
     Signature: You are an expert security researcher and C/C++ programmer.
    Your task is to analyze a C/C++ func...


## Bước 6: Helper — Format Demonstrations và Extract Prediction


In [14]:
def format_demonstrations(demo_list: list) -> str:
    """Chuyển list demonstrations thành text cho DSPy input."""
    if not demo_list:
        return "No similar examples available."
    parts = []
    for i, demo in enumerate(demo_list[:2], 1):  # Tối đa 2 examples
        code_str = demo.get("code", "")[:1500]
        label = demo.get("label", "Unknown")
        parts.append(f"--- Example {i} ({label}) ---\n{code_str}")
    return "\n\n".join(parts)


def extract_prediction_from_verdict(verdict_text: str) -> int:
    """Trích xuất nhãn từ verdict text của DSPy output."""
    text_lower = verdict_text.strip().lower()
    if "non-vulnerable" in text_lower or "non vulnerable" in text_lower:
        return 0
    elif "vulnerable" in text_lower:
        return 1
    # Fallback: keyword matching
    if "0" in verdict_text.strip():
        return 0
    elif "1" in verdict_text.strip():
        return 1
    return 2  # Unknown


def build_dspy_example(item: dict) -> dspy.Example:
    """Tạo dspy.Example từ 1 item trong dataset."""
    demos_text = format_demonstrations(item.get("demonstrations", []))
    return dspy.Example(
        code=item["func"],
        graph_nodes=item.get("node", ""),
        graph_edges=item.get("edge", ""),
        demonstrations=demos_text,
        verdict="Vulnerable" if item["target"] == 1 else "Non-vulnerable",  # gold label
    ).with_inputs("code", "graph_nodes", "graph_edges", "demonstrations")


# Tạo trainset (val set) cho GEPA
print("Chuẩn bị dspy.Example cho val set...")
trainset = [build_dspy_example(item) for item in val_with_demos]
print(f"[OK] Trainset (for GEPA): {len(trainset)} examples")

# Verify 1 example
ex = trainset[0]
print(f"\nExample 0:")
print(f"  code[:80]: {ex.code[:80]}...")
print(f"  graph_nodes[:50]: {ex.graph_nodes[:50] if ex.graph_nodes else '(empty)'}...")
print(f"  demonstrations[:80]: {ex.demonstrations[:80]}...")
print(f"  verdict (gold): {ex.verdict}")


Chuẩn bị dspy.Example cho val set...
[OK] Trainset (for GEPA): 200 examples

Example 0:
  code[:80]: static CharDriverState *qemu_chr_open_socket_fd(int fd, bool do_nodelay,

      ...
  graph_nodes[:50]: (empty)...
  demonstrations[:80]: --- Example 1 (Vulnerable) ---
static CharDriverState *qemu_chr_open_socket(Qemu...
  verdict (gold): Non-vulnerable


## Bước 7: CWE-aware Metric (Phản chiếu theo CWE)

Đây là phần quan trọng nhất của GEPA — hàm metric phải trả về **cả điểm số lẫn feedback dạng text**.

GEPA dùng LLM (reflection_lm) để đọc feedback này và đề xuất cải tiến prompt.

### CWE-aware Feedback:
Khi LLM phân loại sai, feedback sẽ cố gắng hint loại lỗ hổng (CWE category) mà LLM bỏ sót
dựa trên từ khóa trong source code.

### Hai objectives (Pareto):
- **Obj 1:** F1 score (quan tâm đến lớp Vulnerable — minority class)
- **Obj 2:** MCC (Matthew's Correlation Coefficient — tổng hợp cả 2 lớp)


In [15]:
# === Mapping CWE keywords — phát hiện loại lỗ hổng tiềm ẩn ===
CWE_HINTS = {
    "CWE-119 Buffer Overflow": ["strcpy", "strcat", "sprintf", "gets", "memcpy",
                                 "buffer", "array", "overflow", "stack"],
    "CWE-416 Use After Free": ["free(", "use after", "dangling", "heap"],
    "CWE-476 NULL Dereference": ["null", "nullptr", "dereference", "->", "*("],
    "CWE-190 Integer Overflow": ["int", "overflow", "wrap", "unsigned", "size_t"],
    "CWE-362 Race Condition": ["thread", "mutex", "lock", "concurrent", "shared"],
    "CWE-20 Improper Input Validation": ["input", "validate", "sanitize", "user", "argv"],
    "CWE-401 Memory Leak": ["malloc", "calloc", "realloc", "new ", "alloc"],
    "CWE-125 Out-of-Bounds Read": ["index", "offset", "bound", "read", "array["],
}

def get_cwe_hints(code_str: str) -> list:
    """Lấy list CWE types có khả năng liên quan dựa trên code keywords."""
    code_lower = code_str.lower()
    found = []
    for cwe, keywords in CWE_HINTS.items():
        if any(kw in code_lower for kw in keywords):
            found.append(cwe)
    return found[:3]  # Top 3 gợi ý


def cwe_aware_metric(gold: dspy.Example, pred: dspy.Prediction, trace=None, pred_name=None, pred_trace=None) -> dspy.Prediction:
    """
    CWE-aware metric cho GEPA (C3).
    
    Returns:
        dspy.Prediction với:
        - score: float [0, 1] — điểm chính (F1-weighted với MCC)
        - feedback: str — nhận xét tự nhiên cho GEPA reflection
    """
    # === Lấy ground truth và prediction ===
    gold_label = gold.verdict  # "Vulnerable" hoặc "Non-vulnerable"
    pred_verdict = getattr(pred, "verdict", "")
    
    # Extract binary labels
    gold_int = 1 if "non" not in gold_label.lower() else 0
    pred_int = extract_prediction_from_verdict(pred_verdict)
    
    is_correct = (gold_int == pred_int) if pred_int != 2 else False
    
    # === Score component 1: Correctness ===
    base_score = 1.0 if is_correct else 0.0
    
    # Bonus: Vulnerable correctly identified (FN penalization)
    if gold_int == 1 and pred_int == 1:
        base_score = 1.0    # True Positive
    elif gold_int == 1 and pred_int == 0:
        base_score = -0.5   # False Negative — phat nặng hơn (lỗ hổng bỏ sót nguy hiểm)
    elif gold_int == 0 and pred_int == 1:
        base_score = 0.0    # False Positive
    elif gold_int == 0 and pred_int == 0:
        base_score = 1.0    # True Negative
    else:
        base_score = -0.3   # Invalid prediction
    
    # Normalize to [0, 1]
    score = (base_score + 0.5) / 1.5
    score = max(0.0, min(1.0, score))
    
    # === Feedback component: CWE-aware text ===
    code_str = gold.code
    cwe_hints = get_cwe_hints(code_str)
    
    if is_correct:
        feedback = (
            f"Correct prediction: '{pred_verdict}'. "
            f"The model successfully identified the vulnerability status."
        )
        if gold_int == 1 and cwe_hints:
            feedback += f" Relevant CWE categories: {', '.join(cwe_hints)}."
    else:
        if gold_int == 1 and pred_int == 0:
            # False Negative — phai goi y CWE
            feedback = (
                f"WRONG: Predicted 'Non-vulnerable' but ground truth is 'Vulnerable'. "
                f"This is a dangerous False Negative — a real vulnerability was missed. "
            )
            if cwe_hints:
                feedback += (
                    f"Consider that this code may relate to: {', '.join(cwe_hints)}. "
                    f"The prompt should explicitly instruct the model to look for these patterns."
                )
            else:
                feedback += (
                    "The prompt should instruct the model to be more conservative "
                    "and flag suspicious patterns even when not 100% certain."
                )
        elif gold_int == 0 and pred_int == 1:
            feedback = (
                f"WRONG: Predicted 'Vulnerable' but ground truth is 'Non-vulnerable'. "
                f"This is a False Positive. The prompt may be over-sensitizing the model. "
                f"Instruct the model to require concrete evidence of exploitable paths, "
                f"not just presence of potentially dangerous functions."
            )
        else:
            feedback = (
                f"INVALID: Model returned '{pred_verdict}' which is not a valid label. "
                f"The prompt must explicitly require output to be EXACTLY "
                f"'Vulnerable' or 'Non-vulnerable', nothing else."
            )
    
    return dspy.Prediction(score=score, feedback=feedback)


print("[OK] CWE-aware metric defined")

# === Test metric trên 1 ví dụ giả ===
test_gold = dspy.Example(
    code="char buf[10]; strcpy(buf, input);",
    verdict="Vulnerable"
).with_inputs("code")

test_pred_wrong = dspy.Prediction(verdict="Non-vulnerable")
test_pred_right = dspy.Prediction(verdict="Vulnerable")

result_wrong = cwe_aware_metric(test_gold, test_pred_wrong)
result_right = cwe_aware_metric(test_gold, test_pred_right)

print(f"\nTest metric (wrong prediction):")
print(f"  Score: {result_wrong.score:.3f}")
print(f"  Feedback: {result_wrong.feedback}")
print(f"\nTest metric (correct prediction):")
print(f"  Score: {result_right.score:.3f}")
print(f"  Feedback: {result_right.feedback}")


[OK] CWE-aware metric defined

Test metric (wrong prediction):
  Score: 0.000
  Feedback: WRONG: Predicted 'Non-vulnerable' but ground truth is 'Vulnerable'. This is a dangerous False Negative — a real vulnerability was missed. Consider that this code may relate to: CWE-119 Buffer Overflow, CWE-20 Improper Input Validation. The prompt should explicitly instruct the model to look for these patterns.

Test metric (correct prediction):
  Score: 1.000
  Feedback: Correct prediction: 'Vulnerable'. The model successfully identified the vulnerability status. Relevant CWE categories: CWE-119 Buffer Overflow, CWE-20 Improper Input Validation.


## Bước 8: Sanity Check — Chạy thử detector trước khi optimize

Kiểm tra pipeline DSPy hoạt động đúng với Vertex AI trước khi chạy GEPA
(GEPA sẽ gọi LLM rất nhiều lần — muốn chắc chắn không có lỗi trước).


In [16]:
# Thử detector trên 1 example từ val set
sample = val_with_demos[0]
demos_text = format_demonstrations(sample.get("demonstrations", []))

print("=== Sanity Check ===")
print(f"Code (first 200 chars): {sample['func'][:200]}...")
print(f"Ground truth: {'Vulnerable' if sample['target'] == 1 else 'Non-vulnerable'}")
print(f"Demonstrations: {demos_text[:100]}...")
print()

try:
    pred = detector(
        code=sample["func"],
        graph_nodes=sample.get("node", ""),
        graph_edges=sample.get("edge", ""),
        demonstrations=demos_text,
    )
    print(f"Reasoning: {pred.reasoning[:300]}...")
    print(f"Verdict: {pred.verdict}")
    pred_int = extract_prediction_from_verdict(pred.verdict)
    print(f"Predicted label: {pred_int} (GT: {sample['target']})")
    print("[OK] Sanity check passed!")
except Exception as e:
    print(f"[ERROR] {e}")
    print("Kiểm tra lại cau hinh Vertex AI va VERTEX_PROJECT.")


=== Sanity Check ===
Code (first 200 chars): static CharDriverState *qemu_chr_open_socket_fd(int fd, bool do_nodelay,

                                                bool is_listen, bool is_telnet,

                                             ...
Ground truth: Non-vulnerable
Demonstrations: --- Example 1 (Vulnerable) ---
static CharDriverState *qemu_chr_open_socket(QemuOpts *opts)

{

    ...

Reasoning: The function `qemu_chr_open_socket_fd` is responsible for initializing a character device backend in QEMU based on an existing file descriptor. It handles both listening (server) and connected (client) sockets, including support for TCP, IPv6, and Unix domain sockets.

### Key Observations:

1. **In...
Verdict: Vulnerable
Predicted label: 1 (GT: 0)
[OK] Sanity check passed!


## Bước 9: Chạy GEPA Optimization

**GEPA** (Genetic-Pareto Evolutionary Prompt Optimization) sẽ:
1. Chạy detector trên `trainset` (val set) và thu thập traces
2. Dùng `reflection_lm` để đọc feedback từ metric và đề xuất cải tiến instructions
3. Duy trì một "population" các candidate prompts
4. Chọn lọc theo Pareto frontier (score + diversity)
5. Lặp lại cho đến khi đạt `max_rounds`

### Cấu hình tối ưu:
- `auto="medium"`: Preset cân bằng giữa chất lượng và chi phí API (~50-100 calls)
- `max_rounds=5`: Số vòng tiến hoá
- `reflection_lm`: Model mạnh để đọc trace và đề xuất cải tiến

> ⚠️ **Cảnh báo:** Bước này sẽ gọi Vertex AI nhiều lần (~100-200 API calls với preset medium).
> Ước tính thời gian: 30-90 phút tùy tốc độ API.
> Có thể giảm bằng cách set `MAX_VAL_SAMPLES = 50` hoặc `auto="light"`.


In [17]:
# === Chạy GEPA Optimization ===
compiled_cache = "optimized_prompt.json"

if os.path.exists(compiled_cache):
    print(f"[INFO] Đã có compiled program tại {compiled_cache}")
    print("Load lại và bỏ qua optimization. Xóa file này để chạy lại GEPA.")
    compiled_detector = GRACEEnhancedDetector()
    compiled_detector.load(compiled_cache)
    print("[OK] Loaded optimized program")
else:
    print("=== Bắt đầu GEPA Optimization ===")
    print(f"Trainset size: {len(trainset)}")
    print(f"Reflection LM: {QWEN_MODEL}")
    print(f"Auto preset: medium (~100 API calls)")
    print()

    optimizer = dspy.GEPA(
        metric=cwe_aware_metric,
        auto="medium",               # 'light' / 'medium' / 'heavy'
        reflection_lm=reflection_lm,
    )

    compiled_detector = optimizer.compile(
        student=GRACEEnhancedDetector(),
        trainset=trainset,
    )

    # Lưu compiled program
    compiled_detector.save(compiled_cache)
    print(f"\n[OK] Saved optimized program to {compiled_cache}")

# In optimized instructions
print("\n=== Optimized Instructions ===")
for name, param in compiled_detector.named_parameters():
    if hasattr(param, "instructions"):
        print(f"[{name}] Instructions:")
        print(f"  {param.instructions}")


2026/08/01 05:55:39 INFO dspy.teleprompt.gepa.gepa: Running GEPA for approx 1690 metric calls of the program. This amounts to 8.45 full evals on the train set.
2026/08/01 05:55:39 WARNING dspy.teleprompt.gepa.gepa: No valset provided; Using trainset as valset. This is useful as an inference-time scaling strategy where you want GEPA to find the best solutions for the provided tasks in the trainset, as it makes GEPA overfit prompts to the provided trainset. In order to ensure generalization and perform well on unseen tasks, please provide separate trainset and valset. Provide the smallest valset that is just large enough to match the downstream task distribution, while keeping trainset as large as possible.
2026/08/01 05:55:39 INFO dspy.teleprompt.gepa.gepa: Using 200 examples for tracking Pareto scores. You can consider using a smaller sample of the valset to allow GEPA to explore more diverse solutions within the same budget. GEPA requires you to provide the smallest valset that is jus

=== Bắt đầu GEPA Optimization ===
Trainset size: 200
Reflection LM: publishers/qwen/models/qwen3-coder-480b-a35b-instruct-maas
Auto preset: medium (~100 API calls)



GEPA Optimization:   0%|          | 0/1690 [00:00<?, ?rollouts/s]2026/08/01 05:57:57 INFO dspy.evaluate.evaluate: Average Metric: 122.66666666666663 / 200 (61.3%)
2026/08/01 05:57:57 INFO dspy.teleprompt.gepa.gepa: Iteration 0: Base program full valset score: 0.6133333333333334 over 200 / 200 examples
GEPA Optimization:  12%|█▏        | 200/1690 [02:17<17:05,  1.45rollouts/s]2026/08/01 05:57:57 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Selected program 0 score: 0.6133333333333334


Average Metric: 3.00 / 3 (100.0%): 100%|██████████| 3/3 [00:00<00:00, 26.79it/s]

2026/08/01 05:57:57 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/08/01 05:57:57 INFO dspy.teleprompt.gepa.gepa: Iteration 1: All subsample scores perfect. Skipping.
2026/08/01 05:57:57 INFO dspy.teleprompt.gepa.gepa: Iteration 1: Reflective mutation did not propose a new candidate
GEPA Optimization:  12%|█▏        | 203/1690 [02:17<16:42,  1.48rollouts/s]2026/08/01 05:57:57 INFO dspy.teleprompt.gepa.gepa: Iteration 2: Selected program 0 score: 0.6133333333333334


...

2026/08/01 06:15:19 INFO dspy.evaluate.evaluate: Average Metric: 3.0 / 3 (100.0%)
2026/08/01 06:15:19 INFO dspy.teleprompt.gepa.gepa: Iteration 21: All subsample scores perfect. Skipping.
2026/08/01 06:15:19 INFO dspy.teleprompt.gepa.gepa: Iteration 21: Reflective mutation did not propose a new candidate
2026/08/01 06:15:19 INFO dspy.teleprompt.gepa.gepa: Iteration 22: Selected program 5 score: 0.48833333333333334



Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:00<00:00, 39.03it/s] 

2026/08/01 06:15:19 INFO dspy.evaluate.evaluate: Average Metric: 2.0 / 3 (66.7%)


2026/08/01 06:15:23 INFO dspy.teleprompt.gepa.gepa: Iteration 22: Proposed new text for detect.predict: You are a security analyst tasked with identifying vulnerabilities in C code functions. Your goal is to analyze code snippets and determine whether they contain exploitable vulnerabilities (Vulnerable) or not (Non-vulnerable).

**TASK DESCRIPTION:**
Analyze the provided C function code and determine if it contains a security vulnerability that could be exploited. Consider the following vulnerability types:
- NULL pointer dereference (CWE-476)
- Buffer overflow (CWE-119, CWE-121, CWE-787)
- Integer overflow (CWE-190)
- Improper input validation (CWE-20)
- Out-of-bounds read/write
- Use-after-free

**ANALYSIS REQUIREMENTS:**

1. **Require Concrete Evidence**: Only flag functions as "Vulnerable" when you can identify a concrete, exploitable path. Do not flag based on mere presence of potentially dangerous patterns.

2. **Context-Aware Analysis**: 
   - In hardware emulation/embedded sys


[OK] Saved optimized program to optimized_prompt.json

=== Optimized Instructions ===


## Bước 10: Evaluate trên Test Set với Optimized Prompt

Dùng **compiled_detector** (với optimized prompt) để chạy inference trên test set.

Test set đã có **Stratified Demonstrations** từ `setup_and_run_wsl.ipynb` (C1 + C2),
kết hợp với **Optimized Prompt** từ GEPA (C3).


In [25]:
import csv
import math
import os
from sklearn.metrics import matthews_corrcoef, accuracy_score, precision_score, recall_score, f1_score
from tqdm.auto import tqdm

def compute_all_metrics(preds, truths):
    valid_pairs = [(p, t) for p, t in zip(preds, truths) if p in (0, 1)]
    if not valid_pairs:
        return {}
    preds_v, truths_v = zip(*valid_pairs)
    return {
        "Accuracy": accuracy_score(truths_v, preds_v),
        "Precision": precision_score(truths_v, preds_v, zero_division=0),
        "Recall": recall_score(truths_v, preds_v, zero_division=0),
        "F1": f1_score(truths_v, preds_v, zero_division=0),
        "MCC": matthews_corrcoef(truths_v, preds_v),
        "Valid%": len(valid_pairs) / len(preds) * 100,
    }

# === Chạy inference với optimized prompt (Tích hợp Checkpoint) ===
RESULTS_PATH = "devignresults_gepa.csv"
LOG_PATH = "devignmetrics_gepa.log"

predictions = []
ground_truths = []
start_idx = 0

# 1. Đọc checkpoint nếu file CSV đã tồn tại
if os.path.exists(RESULTS_PATH):
    print(f"Đã tìm thấy file {RESULTS_PATH}, đang khôi phục tiến độ...")
    with open(RESULTS_PATH, "r", encoding="utf-8") as csvf:
        reader = csv.reader(csvf)
        next(reader, None)  # Bỏ qua dòng tiêu đề
        for row in reader:
            if row:
                predictions.append(int(row[0]))
                ground_truths.append(int(row[1]))
    start_idx = len(predictions)
    print(f"Đã khôi phục xong {start_idx} mẫu. Chạy tiếp từ mẫu thứ {start_idx + 1}...")
else:
    # Nếu chưa chạy bao giờ thì tạo file và ghi tiêu đề
    with open(RESULTS_PATH, "w", newline="", encoding="utf-8") as csvf:
        writer = csv.writer(csvf)
        writer.writerow(["Prediction", "Groundtruth", "Verdict", "Reasoning"])

print("=== Evaluation: GRACE Enhanced + GEPA Optimized Prompt ===")
print(f"Tổng số test samples: {len(test_processed)}")
print()

# 2. Mở file chế độ 'a' (append - ghi tiếp)
with open(RESULTS_PATH, "a", newline="", encoding="utf-8") as csvf, open(LOG_PATH, "a", encoding="utf-8") as logf:
    writer = csv.writer(csvf)
    
    # Chỉ lặp các mẫu còn lại chưa chạy
    for i in tqdm(range(start_idx, len(test_processed)), desc="Inference", initial=start_idx, total=len(test_processed)):
        item = test_processed[i]
        demos_text = format_demonstrations(item.get("demonstrations", []))

        try:
            with dspy.context(lm=task_lm):
                pred = compiled_detector(
                    code=item["func"],
                    graph_nodes=item.get("node", ""),
                    graph_edges=item.get("edge", ""),
                    demonstrations=demos_text,
                )
            verdict = getattr(pred, "verdict", "")
            reasoning = getattr(pred, "reasoning", "")
        except Exception as e:
            verdict = ""
            reasoning = f"ERROR: {e}"

        pred_int = extract_prediction_from_verdict(verdict)
        predictions.append(pred_int)
        ground_truths.append(item["target"])

        writer.writerow([pred_int, item["target"], verdict[:200], reasoning[:200]])
        # Lệnh flush() này sẽ ghi dữ liệu ngay lập tức xuống đĩa cứng (checkpoint) sau mỗi bước
        csvf.flush()

        if (i + 1) % 10 == 0:
            metrics = compute_all_metrics(predictions, ground_truths)
            msg = (f"[{i+1}/{len(test_processed)}] "
                   f"Acc:{metrics.get('Accuracy', 0):.4f} "
                   f"F1:{metrics.get('F1', 0):.4f} "
                   f"MCC:{metrics.get('MCC', 0):.4f} "
                   f"Valid:{metrics.get('Valid%', 0):.1f}%")
            print(msg)
            logf.write(msg + "\n")
            logf.flush()

print(f"\n[OK] Saved results to {RESULTS_PATH}")

Đã tìm thấy file devignresults_gepa.csv, đang khôi phục tiến độ...
Đã khôi phục xong 2733 mẫu. Chạy tiếp từ mẫu thứ 2734...
=== Evaluation: GRACE Enhanced + GEPA Optimized Prompt ===
Tổng số test samples: 2733



Inference: 100%|██████████| 2733/2733 [00:00<?, ?it/s]


[OK] Saved results to devignresults_gepa.csv


## Bước 11: Bảng So sánh Cuối cùng

So sánh **3 hệ thống**:
1. **Baseline**: Chỉ code + prompt cơ bản (basep.py)
2. **GRACE Enhanced**: Code + CPG Graph + Stratified Demos (C1+C2+C4, llmpre.py)
3. **GRACE+GEPA**: Code + CPG Graph + Stratified Demos + Optimized Prompt (C1+C2+C3+C4)


In [26]:
def load_csv_results(csv_path):
    preds, truths = [], []
    try:
        with open(csv_path, "r") as f:
            reader = csv.DictReader(f)
            for row in reader:
                p = int(row["Prediction"])
                t = int(row["Groundtruth"])
                preds.append(p)
                truths.append(t)
    except FileNotFoundError:
        return None, None
    return preds, truths


experiments = {
    "Baseline (basep)": "devignresults_basep.csv",
    "GRACE Enhanced (C1+C2+C4)": "devignresults_llmpre.csv",
    "GRACE+GEPA (C1+C2+C3+C4)": "devignresults_gepa.csv",
}

all_results = {}
print("Loading results...")
for name, csv_path in experiments.items():
    preds, truths = load_csv_results(csv_path)
    if preds is not None:
        metrics = compute_all_metrics(preds, truths)
        all_results[name] = metrics
        print(f"  [OK] {name}: {len(preds)} predictions")
    else:
        print(f"  [SKIP] {name}: file not found ({csv_path})")

# === Bảng so sánh ===
if all_results:
    print()
    print("=" * 80)
    header = f"{'System':<30} {'Acc':>8} {'P':>8} {'R':>8} {'F1':>8} {'MCC':>8}"
    print(header)
    print("-" * 80)
    
    baseline_metrics = None
    for name, metrics in all_results.items():
        acc = metrics.get("Accuracy", 0)
        p = metrics.get("Precision", 0)
        r = metrics.get("Recall", 0)
        f1 = metrics.get("F1", 0)
        mcc = metrics.get("MCC", 0)
        row = f"{name:<30} {acc:>8.4f} {p:>8.4f} {r:>8.4f} {f1:>8.4f} {mcc:>8.4f}"
        
        if baseline_metrics is None:
            baseline_metrics = metrics
            print(row)
        else:
            # In delta so với baseline
            df1 = f1 - baseline_metrics.get("F1", 0)
            dmcc = mcc - baseline_metrics.get("MCC", 0)
            arrow_f1 = "↑" if df1 > 0 else ("↓" if df1 < 0 else "=")
            arrow_mcc = "↑" if dmcc > 0 else ("↓" if dmcc < 0 else "=")
            print(f"{row}  [F1 {df1:+.4f}{arrow_f1}  MCC {dmcc:+.4f}{arrow_mcc}]")
    
    print("=" * 80)

# === Lưu vào JSON cho báo cáo ===
with open("final_comparison.json", "w") as f:
    json.dump(all_results, f, indent=2)
print(f"\n[OK] Đã lưu bảng kết quả vào final_comparison.json")


Loading results...
  [OK] Baseline (basep): 2733 predictions
  [OK] GRACE Enhanced (C1+C2+C4): 2733 predictions
  [OK] GRACE+GEPA (C1+C2+C3+C4): 2733 predictions

System                              Acc        P        R       F1      MCC
--------------------------------------------------------------------------------
Baseline (basep)                 0.5357   0.4791   0.4897   0.4844   0.0622
GRACE Enhanced (C1+C2+C4)        0.5199   0.4601   0.4503   0.4551   0.0262  [F1 -0.0292↓  MCC -0.0360↓]
GRACE+GEPA (C1+C2+C3+C4)         0.4628   0.4480   0.8838   0.5946   0.0122  [F1 +0.1103↑  MCC -0.0500↓]

[OK] Đã lưu bảng kết quả vào final_comparison.json


## Bước 12: Phân tích Optimized Prompt

In ra **prompt đã được GEPA tối ưu** để hiểu GEPA đã thay đổi gì so với prompt gốc.


In [27]:
print("=== GEPA Optimized Program Analysis ===\n")

# In toàn bộ optimized prompt state
if os.path.exists("optimized_prompt.json"):
    with open("optimized_prompt.json", "r") as f:
        optimized_state = json.load(f)
    
    print("Optimized program state (optimized_prompt.json):")
    print(json.dumps(optimized_state, indent=2)[:3000])
else:
    print("optimized_prompt.json chưa tồn tại (GEPA chưa chạy)")

# In optimized instructions từ compiled_detector
print("\n=== Named Parameters (Optimized Instructions) ===")
for name, param in compiled_detector.named_parameters():
    print(f"\n[{name}]")
    if hasattr(param, "instructions"):
        print(f"  Instructions: {param.instructions}")
    if hasattr(param, "demos"):
        print(f"  Demos: {len(param.demos)} few-shot examples")
        for i, d in enumerate(param.demos[:2]):
            print(f"    Demo {i+1}: {str(d)[:100]}...")


=== GEPA Optimized Program Analysis ===

Optimized program state (optimized_prompt.json):
{
  "detect.predict": {
    "traces": [],
    "train": [],
    "demos": [],
    "signature": {
      "instructions": "You are an expert security researcher and C/C++ programmer.\nYour task is to analyze a C/C++ function and determine if it contains security vulnerabilities.\n\nStudy the provided code carefully, including its graph structure and similar examples.\nReason step by step, then output your final verdict as exactly 'Vulnerable' or 'Non-vulnerable'.\n\nCRITICAL SECURITY PATTERNS TO CHECK FOR:\n1. NULL POINTER DEREFERENCE (CWE-476): Always check if pointers are validated before use, especially function parameters, return values from allocations/functions, and struct members accessed via pointers\n2. INTEGER OVERFLOW (CWE-190): Check for arithmetic operations on integers that could wrap around, especially size calculations, loop counters, and array indexing\n3. MEMORY LEAK (CWE-401): Look f

## Bước 13: Kiểm tra Lịch sử LLM Calls

DSPy lưu lại lịch sử các LM calls. Ta có thể inspect để debug và hiểu cách model trả lời.


In [28]:
# Inspect last few LM calls
print("=== Last 3 LM History Entries ===")
history = task_lm.history
if history:
    for i, entry in enumerate(history[-3:], 1):
        print(f"\n--- Entry {i} ---")
        if isinstance(entry, dict):
            messages = entry.get("messages", [])
            for msg in messages[:2]:
                role = msg.get("role", "?")
                content = str(msg.get("content", ""))[:300]
                print(f"  [{role}]: {content}...")
        print()
else:
    print("No history available yet (chạy sanity check hoặc inference trước)")


=== Last 3 LM History Entries ===

--- Entry 1 ---


--- Entry 2 ---


--- Entry 3 ---

